## Exploratory / untested — gridMET via THREDDS + raster QA scratch

Two independent scratch cells, kept for future reference:
1. `fetch_gridmet_thredds()` — an alternative to the GEE-based gridMET pull used in `notebooks/02_load_flux_openet_gridmet.ipynb`, streaming NetCDF directly from the Northwest Knowledge Network THREDDS server (no GEE account needed).
2. A raw-vs-cached raster QA check for `Data/wbm_rasters/water_storage.tif` (units/scaling and nodata-masking debug).

**Not tested or troubleshot** — kept for future exploration, not part of the main pipeline.

In [ ]:
import sys
from pathlib import Path
_REPO_ROOT = Path.cwd().parent.parent
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))
import pandas as pd  # used by fetch_gridmet_thredds() below


In [ ]:
# NOT TESTED OR TROUBLESHOT BUT claude provided alternative to GEE - to explore in future.



# ## Method B — THREDDS / OPeNDAP  *(alternative to GEE)*
#
# Streams gridded NetCDF files directly from the Northwest Knowledge Network
# THREDDS server using `xarray`.  No GEE account needed.
#
# The aggregated CONUS files cover 1979–present, so we slice the time
# axis to 2016–2023 server-side, then select the nearest grid cell to
# each point — minimising data transfer.
#
# **Install dependencies (once):**
# ```bash
# pip install xarray netCDF4 scipy
# ```

# %%
# ── B1: THREDDS variable catalogue ────────────────────────────────────────────

THREDDS_BASE = (
    "http://thredds.northwestknowledge.net:8080/thredds/dodsC/"
    "agg_met_{var}_1979_CurrentYear_CONUS.nc"
)

# Maps GridMET band name → (THREDDS var name, NetCDF variable inside the file)
GRIDMET_VARS = {
    "pr"  : ("pr",   "precipitation_amount"),   # mm day⁻¹
    "tmmx": ("tmmx", "air_temperature"),         # K
    "tmmn": ("tmmn", "air_temperature"),         # K
    "pet" : ("pet",  "potential_evapotranspiration"),  # mm day⁻¹
    "etr" : ("etr",  "potential_evapotranspiration"),  # mm day⁻¹
}

print("THREDDS URLs that will be opened:")
for band, (var, _) in GRIDMET_VARS.items():
    print(f"  {band:5s}: {THREDDS_BASE.format(var=var)}")

# %%
# ── B2: Fetch one variable for all points and years ───────────────────────────

import xarray as xr
import warnings

def fetch_gridmet_thredds(band: str,
                           towers: pd.DataFrame,
                           start: str,
                           end: str) -> pd.DataFrame:
    """
    Fetch a single GridMET band from THREDDS for all tower points.

    Parameters
    ----------
    band    : GridMET band identifier (e.g. 'pr', 'tmmx').
    towers  : DataFrame with columns x (lon), y (lat), site.
    start   : ISO date string, e.g. '2016-01-01'.
    end     : ISO date string, e.g. '2023-12-31'.

    Returns
    -------
    DataFrame with columns: date, site, x, y, <band>
    """
    var_name, nc_var = GRIDMET_VARS[band]
    url = THREDDS_BASE.format(var=var_name)

    print(f"  Opening {band} from THREDDS ...")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        ds = xr.open_dataset(url, engine="netcdf4")

    # Slice time range server-side (only requested period is downloaded)
    ds = ds.sel(day=slice(start, end))

    frames = []
    for row in towers.itertuples():
        # Select nearest grid cell to the tower
        ds_pt = ds.sel(lon=row.x, lat=row.y, method="nearest")

        df_pt = ds_pt[[nc_var]].to_dataframe().reset_index()
        df_pt = df_pt.rename(columns={"day": "date", nc_var: band})
        df_pt["site"] = row.site
        df_pt["x"]    = float(ds_pt["lon"])   # actual grid cell centre
        df_pt["y"]    = float(ds_pt["lat"])
        frames.append(df_pt[["date", "site", "x", "y", band]])

    ds.close()
    return pd.concat(frames, ignore_index=True)

# %%
# ── B3: Loop over all bands and merge ─────────────────────────────────────────

print(f"Fetching GridMET from THREDDS: {START_DATE} → {END_DATE}")
print(f"Sites: {flux_towers['site'].tolist()}\n")

band_frames = {}
for band in GRIDMET_VARS:
    band_frames[band] = fetch_gridmet_thredds(band, flux_towers,
                                               START_DATE, END_DATE)
    n = len(band_frames[band])
    print(f"  ✓ {band:5s}: {n:,} rows")

# Merge all bands on (date, site, x, y)
thredds_df = band_frames["pr"].copy()
for band in list(GRIDMET_VARS.keys())[1:]:
    merge_cols = band_frames[band][["date", "site", band]]
    thredds_df = thredds_df.merge(merge_cols, on=["date", "site"])

print(f"\nMerged shape: {thredds_df.shape}")
print(thredds_df.head())

# %%
# ── B4: Unit conversions and rename for WBM ───────────────────────────────────

K_TO_C = 273.15

climate_thredds = (
    thredds_df
    .assign(
        ppt_mm          = lambda d: d["pr"],
        tmax_C          = lambda d: d["tmmx"] - K_TO_C,
        tmin_C          = lambda d: d["tmmn"] - K_TO_C,
        tmean_C         = lambda d: (d["tmmx"] + d["tmmn"]) / 2 - K_TO_C,
        pet_gridmet_mm  = lambda d: d["pet"],
        etr_gridmet_mm  = lambda d: d["etr"],
        GCM             = lambda d: d["site"],
        date            = lambda d: pd.to_datetime(d["date"]),
    )
    .loc[:, ["date", "x", "y", "site", "GCM",
             "ppt_mm", "tmax_C", "tmin_C", "tmean_C",
             "pet_gridmet_mm", "etr_gridmet_mm"]]
    .sort_values(["site", "date"])
    .reset_index(drop=True)
)

print(f"Processed THREDDS climate shape: {climate_thredds.shape}")
print(climate_thredds.head(8).to_string(index=False))

# %% [markdown]
# ---
# ## Cell 2 — Assign final climate object and validate
#
# Point `climate_data` at whichever method you used above, then run the
# validation checks that confirm the DataFrame is ready for the WBM.

# %%
# ── Choose your retrieval method ──────────────────────────────────────────────
# Uncomment exactly ONE of the lines below.

climate_data = climate_gee       # Method A — GEE
# climate_data = climate_thredds   # Method B — THREDDS

# %%
# ── Validation checks ─────────────────────────────────────────────────────────

EXPECTED_COLS = ["date", "x", "y", "GCM", "ppt_mm", "tmax_C", "tmin_C", "tmean_C"]
EXPECTED_DAYS = pd.date_range(START_DATE, END_DATE).size   # 2922 for 2016-2023

issues = []

# 1. Required columns present
missing_cols = [c for c in EXPECTED_COLS if c not in climate_data.columns]
if missing_cols:
    issues.append(f"Missing columns: {missing_cols}")

# 2. Correct number of days per site
days_per_site = (
    climate_data.groupby("site")["date"]
    .nunique()
    .rename("n_days")
)
short_sites = days_per_site[days_per_site < EXPECTED_DAYS]
if not short_sites.empty:
    issues.append(f"Fewer than {EXPECTED_DAYS} days for sites:\n{short_sites}")

# 3. No nulls in key columns
null_counts = climate_data[EXPECTED_COLS].isna().sum()
null_cols   = null_counts[null_counts > 0]
if not null_cols.empty:
    issues.append(f"Null values found:\n{null_cols}")

# 4. Temperature in plausible range (°C)
t_min_obs = climate_data["tmean_C"].min()
t_max_obs = climate_data["tmean_C"].max()
if t_min_obs > 50 or t_max_obs > 80:
    issues.append(
        f"tmean_C looks like it may still be in Kelvin "
        f"(min={t_min_obs:.1f}, max={t_max_obs:.1f})"
    )

# 5. Precipitation non-negative
neg_ppt = (climate_data["ppt_mm"] < 0).sum()
if neg_ppt:
    issues.append(f"{neg_ppt} negative precipitation values")

# Report
if issues:
    print("⚠️  Validation issues found:")
    for iss in issues:
        print(f"   • {iss}")
else:
    print("✓ All validation checks passed.\n")
    print(f"  Sites          : {sorted(climate_data['GCM'].unique())}")
    print(f"  Date range     : {climate_data['date'].min().date()} → "
          f"{climate_data['date'].max().date()}")
    print(f"  Rows total     : {len(climate_data):,}")
    print(f"  Days per site  :\n{days_per_site.to_string()}")
    print()
    print("  Annual means per site:")
    ann = (
        climate_data
        .assign(year=lambda d: d["date"].dt.year)
        .groupby("site")
        .agg(
            ppt_mm_ann  = ("ppt_mm",  "sum"),
            tmean_C_ann = ("tmean_C", "mean"),
            pet_mm_ann  = ("pet_gridmet_mm", "sum"),
        )
        / len(climate_data["date"].dt.year.unique())   # divide by n years
    ).round(1)
    print(ann.to_string())

# %% [markdown]
# ---
# ## Cell 3 — (Optional) Save to disk and reload
#
# Cache the fetched data so you don't need to re-query GEE / THREDDS in
# future sessions.

# %%
import os

CACHE_DIR  = "Data/gridmet_cache"
CACHE_FILE = os.path.join(CACHE_DIR, "gridmet_flux_towers_2016_2023.csv")

os.makedirs(CACHE_DIR, exist_ok=True)

# ── Save ──────────────────────────────────────────────────────────────────────
# Default: CSV (no extra dependencies)
climate_data.to_csv(CACHE_FILE, index=False)
print(f"✓ Saved to {CACHE_FILE}  ({os.path.getsize(CACHE_FILE) / 1e6:.1f} MB)")

# Optional: Parquet (faster I/O, ~4× smaller — needs pyarrow or fastparquet)
#   pip install pyarrow
# CACHE_FILE_PQ = CACHE_FILE.replace(".csv", ".parquet")
# climate_data.to_parquet(CACHE_FILE_PQ, index=False)

# ── Reload (use at the top of any subsequent notebook) ────────────────────────
# climate_data = pd.read_csv(CACHE_FILE, parse_dates=["date"])
# climate_data = pd.read_parquet(CACHE_FILE_PQ)   # if using parquet

# %% [markdown]
# ---
# ## Cell 4 — Ready for WBM
#
# `climate_data` now contains every column the WBM expects.
# The snippet below shows how it connects to `run_nps_wbm_points()`.
# (Assumes the setup notebook has already been run.)

# %%
# ── Quick preview of WBM-ready columns ───────────────────────────────────────
WBM_REQUIRED = ["date", "x", "y", "GCM", "ppt_mm", "tmean_C"]
WBM_OPTIONAL = ["tmax_C", "tmin_C"]   # needed only for Penman-Monteith

print("WBM-ready climate_data columns:")
print(f"  Required : {WBM_REQUIRED}")
print(f"  Optional : {WBM_OPTIONAL}")
print(f"  All cols : {climate_data.columns.tolist()}")
print()
print("Example call (uses settings from 00_nps_wbm_setup.py):\n")
print("""
    # Extract site params from rasters (requires Cell 3 of setup notebook)
    point_params_df = extract_point_params(flux_towers)

    # Run WBM for all sites and the full date range
    wbm_results = run_nps_wbm_points(
        climate_data    = climate_data,
        point_params_df = point_params_df,
        pet_method      = PET_METHOD,
        hock_coef       = HOCK_COEF,
        direct_frac     = DIRECT_FRAC,
        return_rate     = RETURN_RATE,
        pet_mult        = PET_MULT,
        soil_mult       = SOIL_MULT,
        shade_coeff     = SHADE_COEFF,
        t_base          = T_BASE,
        to_inches       = TO_INCHES,
        aggregate       = False,    # False → keep per-site rows
    )
""")

In [ ]:
import rasterio
import numpy as np
from rasterio.transform import rowcol
from wbm.raster_io import _RASTERS  # populated by wbm.load_wbm_rasters()

# ── Step 1: Check raw raster values before and after the ×10 scaling ─────────
# Re-open the raw file to see original values before any processing
soil_path = "../../Data/wbm_rasters/water_storage.tif"

with rasterio.open(soil_path) as src:
    raw_arr  = src.read(1).astype(float)
    nodata   = src.nodata
    raw_crs  = src.crs
    raw_trans = src.transform

    print("Raw soil raster (before any processing):")
    print(f"  CRS         : {raw_crs}")
    print(f"  Shape       : {raw_arr.shape}")
    print(f"  NoData val  : {nodata}")
    print(f"  Raw range   : {np.nanmin(raw_arr):.2f} – {np.nanmax(raw_arr):.2f}")
    print(f"  Units hint  : {'likely cm (expect 5–60)' if np.nanmax(raw_arr) < 200 else 'possibly already mm (expect 50–600)'}")

# ── Step 2: Check cached raster values after load_wbm_rasters() ───────────────
cached_soil = _RASTERS["soil"]
print(f"\nCached soil array (after load_wbm_rasters with ×10 and reproject):")
print(f"  Shape       : {cached_soil.shape}")
print(f"  Range       : {np.nanmin(cached_soil):.2f} – {np.nanmax(cached_soil):.2f}")
print(f"  NaN count   : {np.sum(np.isnan(cached_soil)):,} / {cached_soil.size:,} "
      f"({100*np.mean(np.isnan(cached_soil)):.1f}%)")
print(f"  Expected    : 50–600 mm for CONUS soils")

# ── Step 3: Check nodata handling — are nodata values being masked? ────────────
if nodata is not None:
    # Check if nodata values survived into the cache as large/small numbers
    n_nodata_raw = np.sum(raw_arr == nodata)
    print(f"\nNoData check:")
    print(f"  Raw nodata value       : {nodata}")
    print(f"  Raw nodata pixels      : {n_nodata_raw:,}")
    print(f"  Scaled nodata would be : {nodata * 10:.1f}")
    # Check if that scaled nodata value appears in the cache
    n_nodata_cached = np.sum(cached_soil == nodata * 10)
    print(f"  Scaled nodata in cache : {n_nodata_cached:,}  "
          f"{'⚠️  nodata not masked!' if n_nodata_cached > 0 else '✓ masked correctly'}")

# ── Step 4: Sample the raw raster at each tower point (before reprojection) ───
# Compare raw vs cached values side by side
print("\nPer-site soil comparison (raw file vs cached array):")
print(f"  {'Site':<12}  {'Raw (cm)':>10}  {'Cached (mm)':>12}  "
      f"{'Expected mm':>12}  {'Ratio':>8}")
print(f"  {'-'*60}")

for row in flux_towers.itertuples():
    tx, ty = float(row.x), float(row.y)

    # Sample raw raster — need to transform coords to raster CRS first
    # since raw is in EPSG:5070 (Albers)
    try:
        from pyproj import Transformer
        transformer = Transformer.from_crs("EPSG:4326", str(raw_crs),
                                            always_xy=True)
        tx_proj, ty_proj = transformer.transform(tx, ty)
        r_raw, c_raw = rowcol(raw_trans, tx_proj, ty_proj)
        r_raw = int(np.clip(r_raw, 0, raw_arr.shape[0] - 1))
        c_raw = int(np.clip(c_raw, 0, raw_arr.shape[1] - 1))
        raw_val = raw_arr[r_raw, c_raw]
    except Exception:
        raw_val = np.nan

    # Sample cached raster
    r_c, c_c = rowcol(_RASTERS["soil_transform"], tx, ty)
    r_c = int(np.clip(r_c, 0, cached_soil.shape[0] - 1))
    c_c = int(np.clip(c_c, 0, cached_soil.shape[1] - 1))
    cached_val = cached_soil[r_c, c_c]

    expected_mm = raw_val * 10 if not np.isnan(raw_val) else np.nan
    ratio = cached_val / raw_val if (not np.isnan(raw_val) and raw_val != 0) else np.nan

    print(f"  {row.site:<12}  {raw_val:>10.2f}  {cached_val:>12.2f}  "
          f"{expected_mm:>12.2f}  {ratio:>8.2f}")

# ── Step 5: Check what SWC_Max values actually ended up in point_params_df ────
print("\nSWC_Max in point_params_df:")
print(point_params_df[["site", "SWC_Max", "ecosystem"]]
      .sort_values("SWC_Max")
      .to_string(index=False))

print(f"\nSWC_Max stats:")
print(f"  Min    : {point_params_df['SWC_Max'].min():.1f} mm")
print(f"  Max    : {point_params_df['SWC_Max'].max():.1f} mm")
print(f"  Mean   : {point_params_df['SWC_Max'].mean():.1f} mm")
print(f"  Median : {point_params_df['SWC_Max'].median():.1f} mm")
print(f"\n  Expected range for CONUS: 50–400 mm")
print(f"  Values < 50 mm (suspiciously low): "
      f"{(point_params_df['SWC_Max'] < 50).sum()} sites")
print(f"  Values > 400 mm (suspiciously high): "
      f"{(point_params_df['SWC_Max'] > 400).sum()} sites")

